In [ ]:
# import os
# from pathlib import Path
# import datetime
# from tqdm import tqdm
# from dataclasses import dataclass, asdict
# import polars as pl 
# import numpy as np
# from sklearn.linear_model import ElasticNet, ElasticNetCV, LinearRegression
# from sklearn.preprocessing import StandardScaler
# import kaggle_evaluation.default_inference_server

# import pandas as pd
# from typing import List, Dict, Tuple
# from lightgbm import LGBMRegressor          # LightGBM 回归模型
# from sklearn.metrics import r2_score        # 官方评价指标：R²
# import warnings
# import math
# warnings.filterwarnings("ignore", category=RuntimeWarning)

# # ===========================
# # 常量定义
# # ===========================
# EPS = 1e-8  # 防止除 0
# MAX_POSITION = 2
# MIN_POSITION = 0
# ANNUAL_DAYS = 252
# MIN_SIGNAL = 0.0
# MAX_SIGNAL = 2.0

# # 最重要的简单的映射参数

# SIGNAL_MULTIPLIER = 100.0

# DATA_PATH = Path('/kaggle/input/hull-tactical-market-prediction/')

# # ===========================
# # 策略打分函数（官方 Sharpe 变体）
# # ===========================
# def adjusted_sharpe(solution: pd.DataFrame, submission: pd.DataFrame) -> float:
#     """
#     计算带惩罚项的 Sharpe-like 得分：
#     - solution：真实数据，包含 forward_returns, risk_free_rate
#     - submission：预测仓位，列名 prediction，范围 [0,2]
#     """
#     sol = solution.copy().reset_index(drop=False)
#     sub = submission.copy().reset_index(drop=False)
#     if sol.shape[0] != sub.shape[0]:
#         raise ValueError("solution 与 submission 长度不一致")
#     df = sol.copy()
#     df['position'] = sub['prediction'].astype(float).values
    
#     # 检查仓位范围
#     if df['position'].max() > MAX_POSITION + 1e-9:
#         raise ValueError(f"仓位超过上限 {MAX_POSITION}")
#     if df['position'].min() < MIN_POSITION - 1e-9:
#         raise ValueError(f"仓位低于下限 {MIN_POSITION}")
    
#     # 策略收益：无风险 + 仓位 * 超额收益
#     df['strategy_returns'] = (
#         df['risk_free_rate'] * (1 - df['position']) +
#         df['position'] * df['forward_returns']
#     )
#     excess = df['strategy_returns'] - df['risk_free_rate']
#     cum_excess = float((1.0 + excess).prod())
#     mean_excess = (
#         cum_excess ** (1.0 / len(df)) - 1.0
#         if cum_excess > 0 else -1.0
#     )
#     std_excess = float(df['strategy_returns'].std())
    
#     if std_excess == 0:
#         return 0.0
    
#     # 基本 Sharpe
#     sharpe = mean_excess / (std_excess + EPS) * math.sqrt(ANNUAL_DAYS)
#     strat_vol = float(std_excess * math.sqrt(ANNUAL_DAYS) * 100.0)
    
#     # 市场基准
#     market_excess = df['forward_returns'] - df['risk_free_rate']
#     market_cum = float((1.0 + market_excess).prod())
#     market_mean = (
#         market_cum ** (1.0 / len(df)) - 1.0
#         if market_cum > 0 else -1.0
#     )
#     market_std = float(df['forward_returns'].std())
#     market_vol = float(market_std * math.sqrt(ANNUAL_DAYS) * 100.0)
    
#     # 风险惩罚（波动过大罚）
#     excess_vol_penalty = (
#         1.0 + max(0.0, strat_vol / (market_vol + EPS) - 1.2)
#         if market_vol > 0 else 1.0
#     )
#     # 收益不足惩罚
#     return_gap = max(0.0, (market_mean - mean_excess) * 100.0 * ANNUAL_DAYS)
#     return_penalty = 1.0 + (return_gap ** 2) / 100.0
    
#     score = sharpe / (excess_vol_penalty * return_penalty + EPS)
#     return float(min(score, 1_000_000.0))

# # ===========================
# # RSI 计算
# # ===========================
# def compute_RSI_from_prices(price_series, window=14):
#     """
#     根据价格序列计算 RSI 指标：
#     - 先计算涨跌
#     - 使用 EWM 平滑
#     """
#     delta = price_series.diff()
#     up = delta.clip(lower=0.0)
#     down = -delta.clip(upper=0.0)
#     roll_up = up.ewm(alpha=1/window, adjust=False).mean()
#     roll_down = down.ewm(alpha=1/window, adjust=False).mean()
#     rs = roll_up / (roll_down + EPS)
#     return 100 - 100 / (1 + rs).fillna(50.0)

# # ===========================
# # 训练/测试统一的特征工程（无泄漏版本）
# # ===========================
# def preprocess_and_make_features(df, windows=[5, 10, 21, 42, 63, 126]):
#     """
#     对传入的 df 进行统一处理：
#     - 按 date_id 排序
#     - 去掉 market_forward_excess_returns（泄漏）
#     - 构造 U1, U2
#     - 数值列做 EWM 填补（只用过去数据）
#     - 基于 lagged_forward_returns 构造一系列时序指标
#       注意：这里不直接用 forward_returns，只用其滞后版本和技术指标
#     """
#     df = df.sort_values("date_id").reset_index(drop=True)

#     # train 有，test 没有：防止把泄漏目标当特征
#     if "market_forward_excess_returns" in df.columns:
#         df = df.drop(columns=["market_forward_excess_returns"])

#     # 构造 U1, U2（有相关列时才构造）
#     if {"I2", "I1"}.issubset(df.columns):
#         df["U1"] = df["I2"] - df["I1"]
#     if {"M11", "I2", "I9", "I7"}.issubset(df.columns):
#         df["U2"] = df["M11"] / ((df["I2"] + df["I9"] + df["I7"]) / 3.0)

#     # 将 inf / -inf 替换为 NaN
#     df = df.replace([np.inf, -np.inf], np.nan)

#     # 所有数值列做 EWM 填补（按时间向前，不用未来信息）
#     numeric_cols = df.select_dtypes(include=[np.number]).columns
#     for col in numeric_cols:
#         df[col] = df[col].fillna(df[col].ewm(com=0.5, min_periods=1).mean())

#     # ========= 用 lagged_forward_returns 做技术指标 =========
#     # train 里：你自己构造了 lagged_forward_returns
#     # test 里：官方给了 lagged_forward_returns 列
#     if "lagged_forward_returns" not in df.columns:
#         raise ValueError("df 中缺少 lagged_forward_returns，必须先构造或来自 test.csv")

#     lagged = df["lagged_forward_returns"]
#     price = (1.0 + lagged.fillna(0.0)).cumprod()

#     new_cols = {}

#     # 滞后收益
#     new_cols["ret_lag_1"] = lagged.fillna(0.0)
#     for lag in [2, 3, 5, 10]:
#         new_cols[f"ret_lag_{lag}"] = lagged.shift(lag - 1).fillna(0.0)

#     # 滚动指标
#     for w in windows:
#         new_cols[f"mom_{w}"] = price.pct_change(w).fillna(0.0)
#         new_cols[f"roll_mean_{w}"] = lagged.rolling(w, min_periods=1).mean()
#         new_cols[f"roll_std_{w}"] = lagged.rolling(w, min_periods=1).std().fillna(0.0)
#         new_cols[f"roll_skew_{w}"] = lagged.rolling(w, min_periods=5).skew().fillna(0.0)
#         ma = price.rolling(w, min_periods=1).mean()
#         new_cols[f"bias_ma_{w}"] = (price - ma) / (ma + EPS)

#     # 其他技术指标
#     new_cols["skew_21"] = lagged.rolling(21, min_periods=5).skew().fillna(0.0)
#     new_cols["rsi_14"] = compute_RSI_from_prices(price, window=14)

#     ema12 = price.ewm(span=12, adjust=False).mean()
#     ema26 = price.ewm(span=26, adjust=False).mean()
#     macd = ema12 - ema26
#     macd_signal = macd.ewm(span=9, adjust=False).mean()

#     new_cols["macd"] = macd.fillna(0.0)
#     new_cols["macd_signal"] = macd_signal.fillna(0.0)
#     new_cols["macd_hist"] = (macd - macd_signal).fillna(0.0)

#     new_cols["vol_21"] = lagged.rolling(21, min_periods=1).std().fillna(0.0)
#     new_cols["vol_63"] = lagged.rolling(63, min_periods=1).std().fillna(0.0)
#     new_cols["vol_ratio_63_21"] = new_cols["vol_63"] / (new_cols["vol_21"] + EPS)

#     # new_cols["mom21_over_vol21"] = new_cols["mom_21"] / (new_cols["vol_21"] + EPS)
#     new_cols["macd_x_rsi"] = new_cols["macd"] * (new_cols["rsi_14"] - 50)
#     # new_cols["bias_x_mom"] = new_cols["bias_ma_21"] * new_cols["mom_21"]
#     new_cols["skew_x_vol"] = new_cols["skew_21"] * new_cols["vol_21"]

#     # 前一日收益（显式命名）
#     new_cols["forward_return_prev"] = lagged.fillna(0.0)

#     new_df = pd.DataFrame(new_cols)
#     df = pd.concat([df.reset_index(drop=True), new_df], axis=1)
#     df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

#     return df

# # ===========================
# # 时间序列 CV（本地验证用）
# # ===========================
# def time_series_cv_lgbm(
#     train_df,
#     train_len=2000,
#     val_len=180,
#     step=500,
#     skip_initial=1300,
# ):
#     """
#     使用 LightGBM 做时间序列滚动交叉验证：
#     - 自动调用 preprocess_and_make_features
#     - 切分方式：先丢掉前 skip_initial 条，然后窗口滚动：
#         train : [i : i+train_len)
#         val   : [i+train_len : i+train_len+val_len)
#     - 每折打印：索引范围、date_id 范围、R²、Sharpe
#     """
#     # 先统一特征工程
#     train_df = preprocess_and_make_features(train_df)
#     print("特征工程完成，shape:", train_df.shape)

#     # 特征列：去掉时间和标签（forward_returns, risk_free_rate）
#     feature_cols = [
#         c for c in train_df.columns
#         if c not in ["date_id", "forward_returns", "risk_free_rate"]
#     ]

#     # LightGBM 参数
#     lgbm_params = dict(
#         n_estimators=1000,
#         learning_rate=0.001,
#         num_leaves=64,
#         max_depth=5,
#         random_state=42,
#         n_jobs=-1,
#         verbose=-1,
#     )

#     # 丢掉前面噪声段
#     train_df = train_df.iloc[skip_initial:].reset_index(drop=True)
#     n = len(train_df)
#     print("CV 使用数据 shape:", train_df.shape)

#     scores = []
#     sharpe_scores = []
#     fold_info = []
#     fold_idx = 0
#     start_train = 0

#     # 滚动窗口 CV
#     while True:
#         end_train = start_train + train_len
#         start_val = end_train
#         end_val = start_val + val_len
#         if end_val > n:
#             break

#         df_train_fold = train_df.iloc[start_train:end_train]
#         df_val_fold = train_df.iloc[start_val:end_val]

#         X_train = df_train_fold[feature_cols].values
#         y_train = df_train_fold["forward_returns"].values
#         X_val = df_val_fold[feature_cols].values
#         y_val = df_val_fold["forward_returns"].values

#         model = LGBMRegressor(**lgbm_params)
#         model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="l2")

#         # R²
#         y_pred = model.predict(X_val)
#         r2 = r2_score(y_val, y_pred)

#         # 策略 Sharpe
#         position_raw = 1.0 + SIGNAL_MULTIPLIER * y_pred
#         position = np.clip(position_raw, MIN_SIGNAL, MAX_SIGNAL)
#         submission = pd.DataFrame({"prediction": position}, index=df_val_fold.index)

#         solution = df_val_fold[["forward_returns", "risk_free_rate"]].copy()
#         sharpe_score = adjusted_sharpe(solution, submission)

#         scores.append(r2)
#         sharpe_scores.append(sharpe_score)

#         train_date_start = df_train_fold["date_id"].iloc[0]
#         train_date_end   = df_train_fold["date_id"].iloc[-1]
#         val_date_start   = df_val_fold["date_id"].iloc[0]
#         val_date_end     = df_val_fold["date_id"].iloc[-1]

#         fold_info.append({
#             "fold": fold_idx,
#             "train_range_idx": (start_train, end_train),
#             "val_range_idx": (start_val, end_val),
#             "train_date_range": (int(train_date_start), int(train_date_end)),
#             "val_date_range": (int(val_date_start), int(val_date_end)),
#             "r2": float(r2),
#             "sharpe": float(sharpe_score),
#         })

#         print(
#             f"Fold {fold_idx}: "
#             f"train_idx=[{start_train}:{end_train}), "
#             f"val_idx=[{start_val}:{end_val}), "
#             f"train_date=[{train_date_start} ~ {train_date_end}], "
#             f"val_date=[{val_date_start} ~ {val_date_end}], "
#             f"R2={r2:.6f}, Sharpe={sharpe_score:.6f}"
#         )

#         start_train += step
#         fold_idx += 1

#     print("=" * 70)
#     print(f"Total folds: {fold_idx}")
#     print("R2 scores:", [float(s) for s in scores])
#     print("Sharpe scores:", [float(s) for s in sharpe_scores])

#     if scores:
#         print(f"Mean R2:   {np.mean(scores):.6f}")
#         print(f"Std  R2:   {np.std(scores):.6f}")
#     if sharpe_scores:
#         print(f"Mean Sharpe: {np.mean(sharpe_scores):.6f}")
#         print(f"Std  Sharpe: {np.std(sharpe_scores):.6f}")
#     print("=" * 70)

#     return scores, fold_info















# # ===========================
# # 读取 train，构造 lag 特征，然后做本地 CV
# # ===========================
# train_raw = pd.read_csv("/kaggle/input/hull-tactical-market-prediction/train.csv")
# train_raw = train_raw.sort_values("date_id").reset_index(drop=True)

# # 在训练集上自己构造 3 个 lag 特征，使其和 test 的结构对齐
# train_raw['lagged_forward_returns'] = train_raw["forward_returns"].shift(1)
# train_raw['lagged_risk_free_rate'] = train_raw["risk_free_rate"].shift(1)
# train_raw['lagged_market_forward_excess_returns'] = train_raw["market_forward_excess_returns"].shift(1)

# print("原始训练集 shape:", train_raw.shape)

# # 本地 CV 验证（可选）
# scores, fold_info = time_series_cv_lgbm(
#     train_df=train_raw,
#     train_len=5000,
#     val_len=180,
#     step=500,
#     skip_initial=1300,
# )

# # ===========================
# # 训练最终用于提交的模型（1300:-180）
# # ===========================
# # 1）用同样的特征工程
# train_fe = preprocess_and_make_features(train_raw)

# # 2）切出 1300:-180 这个训练区间
# train_fe_final = train_fe.iloc[1300:-180].reset_index(drop=True)
# print("最终训练模型使用的数据 shape:", train_fe_final.shape)

# # 3）定义特征列（与 CV 一致）
# feature_cols_final = [
#     c for c in train_fe_final.columns
#     if c not in ["date_id", "forward_returns", "risk_free_rate"]
# ]

# X_final = train_fe_final[feature_cols_final].values
# y_final = train_fe_final["forward_returns"].values

# # 方便 predict 时对 test 做填补
# train_feature_medians_final = train_fe_final[feature_cols_final].median()

# lgbm_params_final = dict(
#     n_estimators=5000,
#     learning_rate=0.001,
#     num_leaves=64,
#     max_depth=5,
#     random_state=42,
#     n_jobs=-1,
#     verbose=-1,
# )

# model_final = LGBMRegressor(**lgbm_params_final)
# model_final.fit(X_final, y_final)
# print("最终 LightGBM 模型训练完成，特征数:", len(feature_cols_final))

# # ===========================
# # Kaggle 评测接口：predict(test: pl.DataFrame)
# # 使用和训练完全一致的特征工程（流式）
# # ===========================

# # 全局变量：用于在评测过程中累积 test 历史
# test_history_df = None

# def predict(test: pl.DataFrame) -> float:
#     """
#     Kaggle default_inference_server 调用的 predict 函数：
#     - 输入：Polars DataFrame（通常按时间顺序一行一行进来）
#     - 输出：当前最新一行的仓位 allocation ∈ [0, 2]

#     关键点：
#     - 使用和训练相同的 preprocess_and_make_features
#     - 通过全局 test_history_df 累积所有 test 行，保证 rolling / EWM 只用过去
#     """
#     global test_history_df, model_final, feature_cols_final, train_feature_medians_final

#     # pl → pandas
#     test_pd = test.to_pandas()

#     # 按时间拼接到历史序列中（假设调用顺序就是日期顺序）
#     if test_history_df is None:
#         test_history_df = test_pd.copy()
#     else:
#         test_history_df = pd.concat([test_history_df, test_pd], axis=0, ignore_index=True)

#     # test 已经包含 lagged_forward_returns / lagged_risk_free_rate / lagged_market_forward_excess_returns
#     # 直接复用统一特征工程函数
#     test_fe = preprocess_and_make_features(test_history_df)

#     # 确保 test_fe 里有所有训练时用到的特征列，没有的就用训练集的中位数填一列
#     for col in feature_cols_final:
#         if col not in test_fe.columns:
#             default_val = train_feature_medians_final.get(col, 0.0)
#             test_fe[col] = default_val

#     X_all = test_fe[feature_cols_final].copy()
#     X_all = X_all.replace([np.inf, -np.inf], np.nan)
#     X_all = X_all.fillna(train_feature_medians_final)

#     # 使用最后一行作为当前时刻的特征
#     X_last = X_all.iloc[[-1]].values
#     pred_ret = float(model_final.predict(X_last)[0])

#     # 映射为仓位 allocation
#     position_raw = 1.0 + SIGNAL_MULTIPLIER * pred_ret
#     alloc = float(np.clip(position_raw, MIN_SIGNAL, MAX_SIGNAL))
#     return alloc

In [ ]:
import os
from pathlib import Path
import datetime
from tqdm import tqdm
from dataclasses import dataclass, asdict
import polars as pl 
import numpy as np
from sklearn.linear_model import ElasticNet, ElasticNetCV, LinearRegression
from sklearn.preprocessing import StandardScaler
import kaggle_evaluation.default_inference_server

import pandas as pd
from typing import List, Dict, Tuple
from lightgbm import LGBMRegressor          # LightGBM 回归模型
from sklearn.metrics import r2_score        # 官方评价指标：R²
import warnings
import math
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ===========================
# 常量定义
# ===========================
EPS = 1e-8  # 防止除 0
MAX_POSITION = 2
MIN_POSITION = 0
ANNUAL_DAYS = 252
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

# 最重要的简单的映射参数（会在网格搜索后被覆盖）
SIGNAL_MULTIPLIER = 100.0

DATA_PATH = Path('/kaggle/input/hull-tactical-market-prediction/')

# ===========================
# 策略打分函数（官方 Sharpe 变体）
# ===========================
def adjusted_sharpe(solution: pd.DataFrame, submission: pd.DataFrame) -> float:
    """
    计算带惩罚项的 Sharpe-like 得分：
    - solution：真实数据，包含 forward_returns, risk_free_rate
    - submission：预测仓位，列名 prediction，范围 [0,2]
    """
    sol = solution.copy().reset_index(drop=False)
    sub = submission.copy().reset_index(drop=False)
    if sol.shape[0] != sub.shape[0]:
        raise ValueError("solution 与 submission 长度不一致")
    df = sol.copy()
    df['position'] = sub['prediction'].astype(float).values
    
    # 检查仓位范围
    if df['position'].max() > MAX_POSITION + 1e-9:
        raise ValueError(f"仓位超过上限 {MAX_POSITION}")
    if df['position'].min() < MIN_POSITION - 1e-9:
        raise ValueError(f"仓位低于下限 {MIN_POSITION}")
    
    # 策略收益：无风险 + 仓位 * 超额收益
    df['strategy_returns'] = (
        df['risk_free_rate'] * (1 - df['position']) +
        df['position'] * df['forward_returns']
    )
    excess = df['strategy_returns'] - df['risk_free_rate']
    cum_excess = float((1.0 + excess).prod())
    mean_excess = (
        cum_excess ** (1.0 / len(df)) - 1.0
        if cum_excess > 0 else -1.0
    )
    std_excess = float(df['strategy_returns'].std())
    
    if std_excess == 0:
        return 0.0
    
    # 基本 Sharpe
    sharpe = mean_excess / (std_excess + EPS) * math.sqrt(ANNUAL_DAYS)
    strat_vol = float(std_excess * math.sqrt(ANNUAL_DAYS) * 100.0)
    
    # 市场基准
    market_excess = df['forward_returns'] - df['risk_free_rate']
    market_cum = float((1.0 + market_excess).prod())
    market_mean = (
        market_cum ** (1.0 / len(df)) - 1.0
        if market_cum > 0 else -1.0
    )
    market_std = float(df['forward_returns'].std())
    market_vol = float(market_std * math.sqrt(ANNUAL_DAYS) * 100.0)
    
    # 风险惩罚（波动过大罚）
    excess_vol_penalty = (
        1.0 + max(0.0, strat_vol / (market_vol + EPS) - 1.2)
        if market_vol > 0 else 1.0
    )
    # 收益不足惩罚
    return_gap = max(0.0, (market_mean - mean_excess) * 100.0 * ANNUAL_DAYS)
    return_penalty = 1.0 + (return_gap ** 2) / 100.0
    
    score = sharpe / (excess_vol_penalty * return_penalty + EPS)
    return float(min(score, 1_000_000.0))

# ===========================
# RSI 计算
# ===========================
def compute_RSI_from_prices(price_series, window=14):
    """
    根据价格序列计算 RSI 指标：
    - 先计算涨跌
    - 使用 EWM 平滑
    """
    delta = price_series.diff()
    up = delta.clip(lower=0.0)
    down = -delta.clip(upper=0.0)
    roll_up = up.ewm(alpha=1/window, adjust=False).mean()
    roll_down = down.ewm(alpha=1/window, adjust=False).mean()
    rs = roll_up / (roll_down + EPS)
    return 100 - 100 / (1 + rs).fillna(50.0)

# ===========================
# 训练/测试统一的特征工程（无泄漏版本）
# ===========================
def preprocess_and_make_features(df, windows=[5, 10, 21, 42, 63, 126]):
    """
    对传入的 df 进行统一处理：
    - 按 date_id 排序
    - 去掉 market_forward_excess_returns（泄漏）
    - 构造 U1, U2
    - 数值列做 EWM 填补（只用过去数据）
    - 基于 lagged_forward_returns 构造一系列时序指标
      注意：这里不直接用 forward_returns，只用其滞后版本和技术指标
    """
    df = df.sort_values("date_id").reset_index(drop=True)

    # train 有，test 没有：防止把泄漏目标当特征
    if "market_forward_excess_returns" in df.columns:
        df = df.drop(columns=["market_forward_excess_returns"])

    # 构造 U1, U2（有相关列时才构造）
    if {"I2", "I1"}.issubset(df.columns):
        df["U1"] = df["I2"] - df["I1"]
    if {"M11", "I2", "I9", "I7"}.issubset(df.columns):
        df["U2"] = df["M11"] / ((df["I2"] + df["I9"] + df["I7"]) / 3.0)

    # 将 inf / -inf 替换为 NaN
    df = df.replace([np.inf, -np.inf], np.nan)

    # 所有数值列做 EWM 填补（按时间向前，不用未来信息）
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        # 使用指数加权平均进行缺失值填补（只用历史数据）
        df[col] = df[col].fillna(df[col].ewm(com=0.5, min_periods=1).mean())

    # ========= 用 lagged_forward_returns 做技术指标 =========
    # train 里：自己构造了 lagged_forward_returns
    # test 里：官方给了 lagged_forward_returns 列
    if "lagged_forward_returns" not in df.columns:
        raise ValueError("df 中缺少 lagged_forward_returns，必须先构造或来自 test.csv")

    lagged = df["lagged_forward_returns"]
    # 将滞后收益累乘，构造“伪价格”序列
    price = (1.0 + lagged.fillna(0.0)).cumprod()

    new_cols = {}

    # 滞后收益特征
    new_cols["ret_lag_1"] = lagged.fillna(0.0)
    for lag in [2, 3, 5, 10]:
        new_cols[f"ret_lag_{lag}"] = lagged.shift(lag - 1).fillna(0.0)

    # 各种滚动统计指标
    for w in windows:
        new_cols[f"mom_{w}"] = price.pct_change(w).fillna(0.0)
        new_cols[f"roll_mean_{w}"] = lagged.rolling(w, min_periods=1).mean()
        new_cols[f"roll_std_{w}"] = lagged.rolling(w, min_periods=1).std().fillna(0.0)
        new_cols[f"roll_skew_{w}"] = lagged.rolling(w, min_periods=5).skew().fillna(0.0)
        ma = price.rolling(w, min_periods=1).mean()
        new_cols[f"bias_ma_{w}"] = (price - ma) / (ma + EPS)

    # 其他技术指标
    new_cols["skew_21"] = lagged.rolling(21, min_periods=5).skew().fillna(0.0)
    new_cols["rsi_14"] = compute_RSI_from_prices(price, window=14)

    ema12 = price.ewm(span=12, adjust=False).mean()
    ema26 = price.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    macd_signal = macd.ewm(span=9, adjust=False).mean()

    new_cols["macd"] = macd.fillna(0.0)
    new_cols["macd_signal"] = macd_signal.fillna(0.0)
    new_cols["macd_hist"] = (macd - macd_signal).fillna(0.0)

    new_cols["vol_21"] = lagged.rolling(21, min_periods=1).std().fillna(0.0)
    new_cols["vol_63"] = lagged.rolling(63, min_periods=1).std().fillna(0.0)
    new_cols["vol_ratio_63_21"] = new_cols["vol_63"] / (new_cols["vol_21"] + EPS)

    # 一些交互特征
    new_cols["macd_x_rsi"] = new_cols["macd"] * (new_cols["rsi_14"] - 50)
    new_cols["skew_x_vol"] = new_cols["skew_21"] * new_cols["vol_21"]

    # 前一日收益（显式命名）
    new_cols["forward_return_prev"] = lagged.fillna(0.0)

    # 将新特征合并回原 DataFrame
    new_df = pd.DataFrame(new_cols)
    df = pd.concat([df.reset_index(drop=True), new_df], axis=1)
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    return df

# ===========================
# 时间序列 CV（本地验证用）
# ===========================
def time_series_cv_lgbm(
    train_df,
    train_len=2000,
    val_len=180,
    step=500,
    skip_initial=1300,
    signal_multiplier=None,   # 新增：允许传入不同的 SIGNAL_MULTIPLIER
):
    """
    使用 LightGBM 做时间序列滚动交叉验证：
    - 自动调用 preprocess_and_make_features
    - 切分方式：先丢掉前 skip_initial 条，然后窗口滚动：
        train : [i : i+train_len)
        val   : [i+train_len : i+train_len+val_len)
    - 每折打印：索引范围、date_id 范围、R²、Sharpe
    - signal_multiplier：仓位映射时的放大倍数（如果为 None 则使用全局 SIGNAL_MULTIPLIER）
    """
    # 若未指定，则使用当前全局 SIGNAL_MULTIPLIER
    if signal_multiplier is None:
        signal_multiplier = SIGNAL_MULTIPLIER

    # 先统一特征工程
    train_df = preprocess_and_make_features(train_df)
    print("特征工程完成，shape:", train_df.shape)

    # 特征列：去掉时间和标签（forward_returns, risk_free_rate）
    feature_cols = [
        c for c in train_df.columns
        if c not in ["date_id", "forward_returns", "risk_free_rate"]
    ]

    # LightGBM 参数
    lgbm_params = dict(
        n_estimators=1000,
        learning_rate=0.001,
        num_leaves=64,
        max_depth=5,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )

    # 丢掉前面噪声段
    train_df = train_df.iloc[skip_initial:].reset_index(drop=True)
    n = len(train_df)
    print("CV 使用数据 shape:", train_df.shape)

    scores = []          # 存储各折 R²
    sharpe_scores = []   # 存储各折 Sharpe
    fold_info = []       # 保存每折的详细信息
    fold_idx = 0
    start_train = 0

    # 滚动窗口 CV
    while True:
        end_train = start_train + train_len
        start_val = end_train
        end_val = start_val + val_len
        if end_val > n:
            break

        df_train_fold = train_df.iloc[start_train:end_train]
        df_val_fold = train_df.iloc[start_val:end_val]

        X_train = df_train_fold[feature_cols].values
        y_train = df_train_fold["forward_returns"].values
        X_val = df_val_fold[feature_cols].values
        y_val = df_val_fold["forward_returns"].values

        model = LGBMRegressor(**lgbm_params)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="l2")

        # R²
        y_pred = model.predict(X_val)
        r2 = r2_score(y_val, y_pred)

        # 策略 Sharpe：使用当前 signal_multiplier
        position_raw = 1.0 + signal_multiplier * y_pred
        position = np.clip(position_raw, MIN_SIGNAL, MAX_SIGNAL)
        submission = pd.DataFrame({"prediction": position}, index=df_val_fold.index)

        solution = df_val_fold[["forward_returns", "risk_free_rate"]].copy()
        sharpe_score = adjusted_sharpe(solution, submission)

        scores.append(r2)
        sharpe_scores.append(sharpe_score)

        train_date_start = df_train_fold["date_id"].iloc[0]
        train_date_end   = df_train_fold["date_id"].iloc[-1]
        val_date_start   = df_val_fold["date_id"].iloc[0]
        val_date_end     = df_val_fold["date_id"].iloc[-1]

        fold_info.append({
            "fold": fold_idx,
            "train_range_idx": (start_train, end_train),
            "val_range_idx": (start_val, end_val),
            "train_date_range": (int(train_date_start), int(train_date_end)),
            "val_date_range": (int(val_date_start), int(val_date_end)),
            "r2": float(r2),
            "sharpe": float(sharpe_score),
        })

        print(
            f"Fold {fold_idx}: "
            f"train_idx=[{start_train}:{end_train}), "
            f"val_idx=[{start_val}:{end_val}), "
            f"train_date=[{train_date_start} ~ {train_date_end}], "
            f"val_date=[{val_date_start} ~ {val_date_end}], "
            f"R2={r2:.6f}, Sharpe={sharpe_score:.6f}, "
            f"signal_multiplier={signal_multiplier}"
        )

        start_train += step
        fold_idx += 1

    print("=" * 70)
    print(f"Total folds: {fold_idx}")
    print("R2 scores:", [float(s) for s in scores])
    print("Sharpe scores:", [float(s) for s in sharpe_scores])

    if scores:
        print(f"Mean R2:   {np.mean(scores):.6f}")
        print(f"Std  R2:   {np.std(scores):.6f}")
    if sharpe_scores:
        print(f"Mean Sharpe: {np.mean(sharpe_scores):.6f}")
        print(f"Std  Sharpe: {np.std(sharpe_scores):.6f}")
    print("=" * 70)

    # 返回 R² 列表、Sharpe 列表以及每折详细信息
    return scores, sharpe_scores, fold_info

# ===========================
# 读取 train，构造 lag 特征
# ===========================
train_raw = pd.read_csv("/kaggle/input/hull-tactical-market-prediction/train.csv")
train_raw = train_raw.sort_values("date_id").reset_index(drop=True)

# 在训练集上自己构造 3 个 lag 特征，使其和 test 的结构对齐
train_raw['lagged_forward_returns'] = train_raw["forward_returns"].shift(1)
train_raw['lagged_risk_free_rate'] = train_raw["risk_free_rate"].shift(1)
train_raw['lagged_market_forward_excess_returns'] = train_raw["market_forward_excess_returns"].shift(1)

print("原始训练集 shape:", train_raw.shape)

# ===========================
# 网格搜索 train_len 和 SIGNAL_MULTIPLIER
# ===========================
# 候选的训练窗口长度
train_len_candidates = [1000, 2000, 3000, 4000, 5000]
# 候选的仓位映射放大倍数
signal_multiplier_candidates = [50, 100, 150, 200, 250, 300, 350, 400, 450, 500]

best_sharpe_overall = -np.inf           # 全局最优 Sharpe
best_train_len = None                   # 最优 train_len
best_signal_multiplier = None           # 最优 SIGNAL_MULTIPLIER

# 为了避免在 time_series_cv_lgbm 里重复做特征工程太多次，
# 也可以提前做一次特征工程；考虑到函数内部也会调用，这里保持一致性，直接用原始 train_raw。
for tl in train_len_candidates:
    for sm in signal_multiplier_candidates:
        print(f"\n=== 开始 CV：train_len={tl}, SIGNAL_MULTIPLIER={sm} ===")
        # 调用时间序列 CV，返回 R² 列表、Sharpe 列表和折信息
        scores, sharpe_scores, fold_info = time_series_cv_lgbm(
            train_df=train_raw,
            train_len=tl,
            val_len=180,
            step=500,
            skip_initial=1300,
            signal_multiplier=sm,
        )

        # 如果该参数组合下没有有效折数，则跳过
        if not sharpe_scores:
            continue

        mean_sharpe = float(np.mean(sharpe_scores))
        print(f"参数组合 train_len={tl}, SIGNAL_MULTIPLIER={sm} 的平均 Sharpe={mean_sharpe:.6f}")

        # 更新全局最优
        if mean_sharpe > best_sharpe_overall:
            best_sharpe_overall = mean_sharpe
            best_train_len = tl
            best_signal_multiplier = sm
            print(f"*** 发现新的最优组合：train_len={tl}, SIGNAL_MULTIPLIER={sm}, mean Sharpe={mean_sharpe:.6f} ***")

print("\n" + "#" * 80)
print("网格搜索完成！")
print(f"最优参数组合：train_len={best_train_len}, SIGNAL_MULTIPLIER={best_signal_multiplier}, mean Sharpe={best_sharpe_overall:.6f}")
print("#" * 80 + "\n")

# 用最优的 SIGNAL_MULTIPLIER 更新全局变量，后面训练与 predict 都会用到
SIGNAL_MULTIPLIER = float(best_signal_multiplier)

# ===========================
# 训练最终用于提交的模型（使用最优 train_len）
# ===========================
# 1）用同样的特征工程
train_fe = preprocess_and_make_features(train_raw)

# 与 CV 一致的跳过与保留设置
skip_initial_final = 1300   # 与 CV 中 skip_initial 保持一致
val_len_final = 180         # 与 CV 中 val_len 保持一致（用于截掉末尾）

# 先在特征工程后的数据上截掉前 skip_initial 行和最后 val_len 行
train_window = train_fe.iloc[skip_initial_final:-val_len_final].reset_index(drop=True)

# 根据最优 train_len 从这一段中取最后 best_train_len 行进行训练
if best_train_len is not None and best_train_len < len(train_window):
    train_fe_final = train_window.iloc[-best_train_len:].reset_index(drop=True)
else:
    # 如果候选的 train_len 大于可用样本，则退化为用所有可用数据
    train_fe_final = train_window

print("最终训练模型使用的数据 shape:", train_fe_final.shape)
print(f"最终训练使用的窗口长度（实际样本数）: {len(train_fe_final)}，来自候选 train_len={best_train_len}")

# 3）定义特征列（与 CV 一致）
feature_cols_final = [
    c for c in train_fe_final.columns
    if c not in ["date_id", "forward_returns", "risk_free_rate"]
]

X_final = train_fe_final[feature_cols_final].values
y_final = train_fe_final["forward_returns"].values

# 方便 predict 时对 test 做填补：记录训练集中各特征的中位数
train_feature_medians_final = train_fe_final[feature_cols_final].median()

lgbm_params_final = dict(
    n_estimators=5000,
    learning_rate=0.001,
    num_leaves=64,
    max_depth=5,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

# 训练最终 LightGBM 模型
model_final = LGBMRegressor(**lgbm_params_final)
model_final.fit(X_final, y_final)
print("最终 LightGBM 模型训练完成，特征数:", len(feature_cols_final))
print(f"最终使用的 SIGNAL_MULTIPLIER={SIGNAL_MULTIPLIER}")

# ===========================
# Kaggle 评测接口：predict(test: pl.DataFrame)
# 使用和训练完全一致的特征工程（流式）
# ===========================

# 全局变量：用于在评测过程中累积 test 历史
test_history_df = None

def predict(test: pl.DataFrame) -> float:
    """
    Kaggle default_inference_server 调用的 predict 函数：
    - 输入：Polars DataFrame（通常按时间顺序一行一行进来）
    - 输出：当前最新一行的仓位 allocation ∈ [0, 2]

    关键点：
    - 使用和训练相同的 preprocess_and_make_features
    - 通过全局 test_history_df 累积所有 test 行，保证 rolling / EWM 只用过去
    - 仓位映射使用网格搜索得到的最优 SIGNAL_MULTIPLIER
    """
    global test_history_df, model_final, feature_cols_final, train_feature_medians_final, SIGNAL_MULTIPLIER

    # pl → pandas
    test_pd = test.to_pandas()

    # 按时间拼接到历史序列中（假设调用顺序就是日期顺序）
    if test_history_df is None:
        test_history_df = test_pd.copy()
    else:
        test_history_df = pd.concat([test_history_df, test_pd], axis=0, ignore_index=True)

    # test 已经包含 lagged_forward_returns / lagged_risk_free_rate / lagged_market_forward_excess_returns
    # 直接复用统一特征工程函数
    test_fe = preprocess_and_make_features(test_history_df)

    # 确保 test_fe 里有所有训练时用到的特征列，没有的就用训练集的中位数填一列
    for col in feature_cols_final:
        if col not in test_fe.columns:
            default_val = train_feature_medians_final.get(col, 0.0)
            test_fe[col] = default_val

    X_all = test_fe[feature_cols_final].copy()
    X_all = X_all.replace([np.inf, -np.inf], np.nan)
    X_all = X_all.fillna(train_feature_medians_final)

    # 使用最后一行作为当前时刻的特征
    X_last = X_all.iloc[[-1]].values
    pred_ret = float(model_final.predict(X_last)[0])

    # 映射为仓位 allocation（使用最优 SIGNAL_MULTIPLIER）
    position_raw = 1.0 + SIGNAL_MULTIPLIER * pred_ret
    alloc = float(np.clip(position_raw, MIN_SIGNAL, MAX_SIGNAL))
    return alloc


In [ ]:
1

# 需要循环检测的参数

- 训练数据长度
- 收益率->仓位的缩放方法（非线性）####################
- 在线学习功能添加：（隔步训练使用）
- 如果使用滚动特征，怎么对测试数据上做处理

# 可能会有用的特征：
- forward_returns——lag1，2，3，4，5，6，rollingmean，EMA等
- 测试收益转仓位的粗略以及本地验证：（）
- 构造本地的180天验证对齐排行榜
- 特征筛选

In [ ]:
# ===========================
# 启动 Kaggle 评测服务
# ===========================
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))